In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
!pip install faker
from faker import Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.4 MB/s eta 0:00:00


#`df_personal`

## Data Wrangling

### Gathering Data

In [2]:
# =========================================================
# REALISTIC SYNTHETIC PERSONAL FINANCE DATASET GENERATOR
# Anti Data Leakage Version
# =========================================================

# =========================================================
# INITIALIZATION
# =========================================================

fake = Faker('id_ID')

random.seed(42)
np.random.seed(42)

# =========================================================
# CATEGORY CONFIG
# =========================================================

categories = {
    "Makanan & Minuman": [
        "Makan siang",
        "Makan malam",
        "Ngopi",
        "Jajan",
        "GoFood",
        "GrabFood",
        "Minimarket",
        "Cafe",
        "Restoran"
    ],

    "Transportasi": [
        "Bensin",
        "Tol",
        "Parkir",
        "Gojek",
        "Grab",
        "KRL",
        "MRT"
    ],

    "Belanja Bulanan": [
        "Shopee",
        "Tokopedia",
        "Lazada",
        "Minimarket",
        "Supermarket",
        "Beli pakaian",
        "Peralatan rumah"
    ],

    "Pulsa & Internet": [
        "Pulsa",
        "Paket data",
        "Tagihan internet",
        "Wifi rumah"
    ],

    "Hiburan": [
        "Netflix",
        "Spotify",
        "Bioskop",
        "Konser",
        "Liburan",
        "Game online"
    ],

    "Tempat Tinggal": [
        "Bayar kos",
        "Listrik",
        "Air",
        "Service AC",
        "Iuran lingkungan"
    ],

    "Gaji": [
        "Gaji bulanan",
        "Bonus",
        "THR"
    ],

    "Transfer Teman/Keluarga": [
        "Transfer",
        "Kirim uang",
        "Bayar teman",
        "Patungan"
    ],

    "Pembayaran Langganan": [
        "Cloud storage",
        "Gym membership",
        "Subscription",
        "Membership"
    ]
}

# =========================================================
# TRANSACTION TYPE
# =========================================================

transaction_types = {
    "Pemasukan": [
        "Gaji",
        "Transfer Teman/Keluarga"
    ],

    "Pengeluaran": [
        "Makanan & Minuman",
        "Transportasi",
        "Belanja Bulanan",
        "Pulsa & Internet",
        "Hiburan",
        "Tempat Tinggal",
        "Pembayaran Langganan",
        "Transfer Teman/Keluarga"
    ]
}

# =========================================================
# PAYMENT METHODS
# Tidak deterministic agar tidak leakage
# =========================================================

payment_methods = [
    "Tunai",
    "Kartu Debit",
    "Kartu Kredit",
    "E-Wallet",
    "Transfer Bank"
]

# =========================================================
# AMOUNT GENERATOR
# Menggunakan distribusi statistik
# =========================================================

def generate_amount(category):

    if category == "Makanan & Minuman":
        amount = np.random.normal(50000, 20000)

    elif category == "Transportasi":
        amount = np.random.gamma(2, 15000)

    elif category == "Belanja Bulanan":
        amount = np.random.lognormal(12, 0.5)

    elif category == "Pulsa & Internet":
        amount = np.random.normal(100000, 40000)

    elif category == "Hiburan":
        amount = np.random.lognormal(11.5, 0.7)

    elif category == "Tempat Tinggal":
        amount = np.random.normal(850000, 250000)

    elif category == "Gaji":
        amount = np.random.normal(6500000, 800000)

    elif category == "Transfer Teman/Keluarga":
        amount = np.random.normal(250000, 150000)

    elif category == "Pembayaran Langganan":
        amount = np.random.normal(120000, 60000)

    else:
        amount = np.random.normal(100000, 50000)

    return int(max(10000, amount))

# =========================================================
# TYPO / NOISE GENERATOR
# =========================================================

def add_noise(text):

    typo_prob = 0.15

    if random.random() > typo_prob:
        return text

    typo_dict = {
        "Netflix": "Netfliix",
        "Spotify": "Spotfy",
        "Gojek": "Gojek",
        "Grab": "Grabb",
        "Makan": "Mkn",
        "Transfer": "Trf"
    }

    for k, v in typo_dict.items():
        if k in text:
            return text.replace(k, v)

    return text

# =========================================================
# GENERATE DATASET
# =========================================================

def generate_dataset(num_rows=5000):

    data = []

    start_date = datetime.now() - timedelta(days=730)
    end_date = datetime.now()

    for i in range(num_rows):

        # =================================================
        # Generate transaction type
        # =================================================

        transaction_type = np.random.choice(
            ["Pengeluaran", "Pemasukan"],
            p=[0.85, 0.15]
        )

        # =================================================
        # Generate category
        # =================================================

        category = random.choice(
            transaction_types[transaction_type]
        )

        # =================================================
        # Generate description
        # =================================================

        description = random.choice(
            categories[category]
        )

        # tambahkan ambiguity
        ambiguous_descriptions = [
            "Transfer",
            "Pembayaran digital",
            "Marketplace",
            "Tagihan"
        ]

        if random.random() < 0.10:
            description = random.choice(
                ambiguous_descriptions
            )

        # typo/noise
        description = add_noise(description)

        # =================================================
        # Generate amount
        # =================================================

        amount = generate_amount(category)

        # overlap amount antar kategori
        if random.random() < 0.15:
            amount *= random.uniform(0.5, 1.5)

        amount = int(amount)

        # =================================================
        # Payment method
        # tidak deterministic
        # =================================================

        payment_method = np.random.choice(
            payment_methods,
            p=[0.15, 0.30, 0.15, 0.25, 0.15]
        )

        # =================================================
        # Date generation
        # =================================================

        transaction_date = fake.date_between_dates(
            date_start=start_date,
            date_end=end_date
        )

        # =================================================
        # Add row
        # =================================================

        data.append({

            "transaction_id": i + 1,

            "transaction_date":
                transaction_date.strftime("%Y-%m-%d"),

            "description": description,

            "category": category,

            "transaction_type": transaction_type,

            "amount": amount,

            "payment_method": payment_method
        })

    return pd.DataFrame(data)

# =========================================================
# GENERATE DATASET
# =========================================================

df_personal = generate_dataset(5000)

# =========================================================
# PREVIEW
# =========================================================

display(df_personal.head())

print(df_personal.info())

print("\nJumlah kategori:")
print(df_personal["category"].value_counts())

,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1,2026-04-15,Bensin,Transportasi,Pengeluaran,6024,Tunai
1,2,2024-11-24,THR,Gaji,Pemasukan,6755121,E-Wallet
2,3,2025-05-24,Restoran,Makanan & Minuman,Pengeluaran,40610,Kartu Debit
3,4,2025-06-08,Wifi rumah,Pulsa & Internet,Pengeluaran,158920,Kartu Debit
4,5,2024-11-26,Subscription,Pembayaran Langganan,Pengeluaran,16504,E-Wallet


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    5000 non-null   int64 
 1   transaction_date  5000 non-null   object
 2   description       5000 non-null   object
 3   category          5000 non-null   object
 4   transaction_type  5000 non-null   object
 5   amount            5000 non-null   int64 
 6   payment_method    5000 non-null   object
dtypes: int64(2), object(5)
memory usage: 273.6+ KB
None

Jumlah kategori:
category
Transfer Teman/Keluarga    884
Pembayaran Langganan       583
Transportasi               555
Hiburan                    550
Belanja Bulanan            544
Makanan & Minuman          528
Pulsa & Internet           519
Tempat Tinggal             476
Gaji                       361
Name: count, dtype: int64


In [3]:
# --- Save Datasets to CSV ---
df_personal.to_csv("fintrack_personal_dataset.csv", index=False)
print("\nDatasets saved as 'fintrack_personal_dataset.csv'")


Datasets saved as 'fintrack_personal_dataset.csv'


In [4]:
# =========================================================
# NOISE CONFIGURATION
# =========================================================

missing_percentage = 0.01
duplicate_percentage = 0.05
negative_amount_percentage = 0.02
inconsistency_percentage = 0.02

# =========================================================
# REALISTIC INCONSISTENCY MAP
# lebih realistis dan tidak terlalu absurd
# =========================================================

personal_inconsistency_map = {

    "Netflix": [
        "Pembayaran Langganan",
        "Pulsa & Internet"
    ],

    "Spotify": [
        "Pembayaran Langganan",
        "Hiburan"
    ],

    "GoFood": [
        "Belanja Bulanan",
        "Makanan & Minuman"
    ],

    "GrabFood": [
        "Belanja Bulanan",
        "Makanan & Minuman"
    ],

    "Marketplace": [
        "Belanja Bulanan",
        "Pembayaran Langganan"
    ],

    "Transfer": [
        "Transfer Teman/Keluarga",
        "Belanja Bulanan"
    ],

    "Tagihan": [
        "Pulsa & Internet",
        "Tempat Tinggal"
    ],

    "Subscription": [
        "Hiburan",
        "Pembayaran Langganan"
    ]
}

# =========================================================
# ADD NOISE FUNCTION
# =========================================================

def add_noise_to_dataframe(df, inconsistency_map, all_categories_map):

    df_noisy = df.copy()

    print("Applying noise injection...")

    # =====================================================
    # 1. MISSING VALUES
    # =====================================================

    num_rows = len(df_noisy)
    num_cols = len(df_noisy.columns)

    num_missing = int(
        num_rows * num_cols * missing_percentage
    )

    for _ in range(num_missing):

        row_idx = random.randint(0, num_rows - 1)
        col_idx = random.randint(0, num_cols - 1)

        column_name = df_noisy.columns[col_idx]

        # hindari missing pada target
        if column_name == "category":
            continue

        df_noisy.iat[row_idx, col_idx] = np.nan

    # =====================================================
    # 2. DUPLICATE ROWS
    # =====================================================

    num_duplicates = int(
        len(df_noisy) * duplicate_percentage
    )

    duplicate_rows = df_noisy.sample(
        n=num_duplicates,
        replace=True,
        random_state=42
    )

    df_noisy = pd.concat(
        [df_noisy, duplicate_rows],
        ignore_index=True
    )

    # =====================================================
    # 3. NEGATIVE AMOUNT
    # hanya untuk pengeluaran
    # =====================================================

    if "amount" in df_noisy.columns:

        positive_indices = df_noisy[
            df_noisy["amount"] > 0
        ].index.tolist()

        num_negative = int(
            len(df_noisy) * negative_amount_percentage
        )

        selected_indices = random.sample(
            positive_indices,
            min(num_negative, len(positive_indices))
        )

        for idx in selected_indices:

            if (
                df_noisy.loc[idx, "transaction_type"]
                == "Pengeluaran"
            ):

                df_noisy.loc[idx, "amount"] *= -1

    # =====================================================
    # 4. CATEGORY INCONSISTENCY
    # versi lebih realistis
    # =====================================================

    eligible_indices = []

    for idx, row in df_noisy.iterrows():

        desc = str(row["description"])

        if desc in inconsistency_map:

            eligible_indices.append(idx)

    num_inconsistent = int(
        len(df_noisy) * inconsistency_percentage
    )

    inconsistent_indices = random.sample(
        eligible_indices,
        min(num_inconsistent, len(eligible_indices))
    )

    for idx in inconsistent_indices:

        desc = str(
            df_noisy.loc[idx, "description"]
        )

        current_category = str(
            df_noisy.loc[idx, "category"]
        )

        possible_categories = [
            cat
            for cat in inconsistency_map[desc]
            if cat != current_category
        ]

        # skip jika tidak ada alternatif
        if len(possible_categories) == 0:
            continue

        new_category = random.choice(
            possible_categories
        )

        df_noisy.loc[idx, "category"] = (
            new_category
        )

    # =====================================================
    # 5. TYPO / TEXT NOISE
    # =====================================================

    typo_probability = 0.03

    typo_map = {

        "Netflix": "Netfliix",
        "Spotify": "Spotfy",
        "Gojek": "Gojekk",
        "Marketplace": "Marketplce",
        "Transfer": "Trf",
        "Makan": "Mkn",
        "Subscription": "Subcription"
    }

    for idx in df_noisy.index:

        if random.random() < typo_probability:

            desc = str(
                df_noisy.loc[idx, "description"]
            )

            for correct, typo in typo_map.items():

                if correct in desc:

                    df_noisy.loc[idx, "description"] = (
                        desc.replace(correct, typo)
                    )

    # =====================================================
    # 6. OUTLIER AMOUNT
    # =====================================================

    outlier_percentage = 0.01

    num_outliers = int(
        len(df_noisy) * outlier_percentage
    )

    outlier_indices = random.sample(
        list(df_noisy.index),
        num_outliers
    )

    for idx in outlier_indices:

        current_amount = df_noisy.loc[idx, "amount"]

        # skip jika NaN
        if pd.isna(current_amount):
            continue

        multiplier = random.uniform(3, 8)

        df_noisy.loc[idx, "amount"] = int(
            abs(current_amount) * multiplier
        )

    # =====================================================
    # 7. RANDOM AMOUNT OVERLAP
    # membuat distribusi lebih realistis
    # =====================================================

    overlap_percentage = 0.10

    num_overlap = int(
        len(df_noisy) * overlap_percentage
    )

    overlap_indices = random.sample(
        list(df_noisy.index),
        num_overlap
    )

    for idx in overlap_indices:

        current_amount = df_noisy.loc[idx, "amount"]

        if pd.isna(current_amount):
            continue

        multiplier = random.uniform(0.3, 2.5)

        df_noisy.loc[idx, "amount"] = int(
            abs(current_amount) * multiplier
        )

    print("Noise injection complete.")

    return df_noisy

# =========================================================
# APPLY NOISE
# =========================================================

print("Applying noise to personal dataset...")

df_personal_noisy = add_noise_to_dataframe(
    df_personal,
    personal_inconsistency_map,
    categories
)

# =========================================================
# PREVIEW
# =========================================================

print("\nDataset Preview:")
display(df_personal_noisy.head())

print("\nDataset Info:")
print(df_personal_noisy.info())

print("\nMissing Values:")
print(df_personal_noisy.isnull().sum())

print("\nDuplicate Rows:")
print(df_personal_noisy.duplicated().sum())

print("\nCategory Distribution:")
print(
    df_personal_noisy["category"]
    .value_counts()
)

# =========================================================
# SAVE DATASET
# =========================================================

df_personal_noisy.to_csv(
    "fintrack_personal_noisy_dataset.csv",
    index=False
)

print(
    "\nDataset saved as "
    "'fintrack_personal_noisy_dataset.csv'"
)

Applying noise to personal dataset...
Applying noise injection...
Noise injection complete.

Dataset Preview:


,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1.0,2026-04-15,Bensin,Transportasi,Pengeluaran,6024.0,Tunai
1,2.0,2024-11-24,THR,Gaji,Pemasukan,6755121.0,E-Wallet
2,3.0,2025-05-24,Restoran,Makanan & Minuman,Pengeluaran,40610.0,Kartu Debit
3,4.0,2025-06-08,Wifi rumah,Pulsa & Internet,Pengeluaran,274540.0,Kartu Debit
4,5.0,2024-11-26,Subscription,Pembayaran Langganan,Pengeluaran,16504.0,E-Wallet



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    5208 non-null   float64
 1   transaction_date  5215 non-null   object 
 2   description       5189 non-null   object 
 3   category          5250 non-null   object 
 4   transaction_type  5194 non-null   object 
 5   amount            5195 non-null   float64
 6   payment_method    5197 non-null   object 
dtypes: float64(2), object(5)
memory usage: 287.2+ KB
None

Missing Values:
transaction_id      42
transaction_date    35
description         61
category             0
transaction_type    56
amount              55
payment_method      53
dtype: int64

Duplicate Rows:
191

Category Distribution:
category
Transfer Teman/Keluarga    915
Belanja Bulanan            610
Pembayaran Langganan       591
Transportasi               580
Hiburan                    5

# `df_business`

## Data Wrangling

### Gathering Data

In [5]:
# =========================================================
# REALISTIC SYNTHETIC BUSINESS FINANCE DATASET GENERATOR
# Anti Data Leakage Version
# =========================================================

# =========================================================
# INITIALIZATION
# =========================================================

fake = Faker('id_ID')

random.seed(42)
np.random.seed(42)

# =========================================================
# CATEGORY CONFIG
# =========================================================

business_categories = {

    "Penjualan": [
        "Penjualan",
        "Marketplace",
        "Pendapatan usaha",
        "Pendapatan online"
    ],

    "Modal & Investasi": [
        "Pendanaan usaha",
        "Transfer modal",
        "Pendapatan investasi"
    ],

    "Piutang": [
        "Pembayaran pelanggan",
        "Pelunasan",
        "Transfer masuk"
    ],

    "Pembelian Stok": [
        "Pembelian barang",
        "Restock",
        "Kebutuhan usaha"
    ],

    "Operasional Kantor": [
        "Tagihan kantor",
        "Biaya operasional",
        "Kebutuhan kantor"
    ],

    "Gaji & Karyawan": [
        "Payroll",
        "Bonus",
        "Pembayaran karyawan"
    ],

    "Marketing & Promosi": [
        "Promosi",
        "Iklan",
        "Campaign"
    ],

    "Transportasi & Logistik": [
        "Pengiriman",
        "Transportasi",
        "Operasional kendaraan"
    ],

    "Peralatan & Aset": [
        "Pembelian aset",
        "Peralatan bisnis",
        "Maintenance"
    ],

    "Software & Langganan": [
        "Subscription",
        "Cloud service",
        "Software bisnis"
    ],

    "Pajak & Perizinan": [
        "Pajak",
        "Administrasi legal",
        "Perizinan"
    ],

    "Utang & Cicilan": [
        "Pembayaran cicilan",
        "Pelunasan utang",
        "Pembayaran pinjaman"
    ],

    "Biaya Bank": [
        "Biaya transaksi",
        "Biaya administrasi",
        "Payment gateway"
    ],

    "Lain-lain": [
        "Refund",
        "Koreksi transaksi",
        "Biaya tak terduga"
    ]
}

# =========================================================
# TRANSACTION TYPE
# =========================================================

business_transaction_types = {

    "Pemasukan": [
        "Penjualan",
        "Modal & Investasi",
        "Piutang"
    ],

    "Pengeluaran": [
        "Pembelian Stok",
        "Operasional Kantor",
        "Gaji & Karyawan",
        "Marketing & Promosi",
        "Transportasi & Logistik",
        "Peralatan & Aset",
        "Software & Langganan",
        "Pajak & Perizinan",
        "Utang & Cicilan",
        "Biaya Bank",
        "Lain-lain"
    ]
}

# =========================================================
# PAYMENT METHODS
# Tidak deterministic agar tidak leakage
# =========================================================

business_payment_methods = [

    "Tunai",
    "Kartu Debit",
    "Kartu Kredit",
    "E-Wallet",
    "Transfer Bank",
    "Payment Gateway",
    "QRIS"
]

# =========================================================
# AMOUNT GENERATOR
# Menggunakan distribusi statistik
# =========================================================

def generate_business_amount(category):

    if category == "Penjualan":
        amount = np.random.lognormal(14, 0.8)

    elif category == "Modal & Investasi":
        amount = np.random.lognormal(16, 0.9)

    elif category == "Piutang":
        amount = np.random.lognormal(14.5, 0.7)

    elif category == "Pembelian Stok":
        amount = np.random.lognormal(14.2, 0.7)

    elif category == "Operasional Kantor":
        amount = np.random.normal(3000000, 1500000)

    elif category == "Gaji & Karyawan":
        amount = np.random.normal(7000000, 3000000)

    elif category == "Marketing & Promosi":
        amount = np.random.lognormal(13.8, 0.9)

    elif category == "Transportasi & Logistik":
        amount = np.random.gamma(2, 500000)

    elif category == "Peralatan & Aset":
        amount = np.random.lognormal(15, 1)

    elif category == "Software & Langganan":
        amount = np.random.normal(800000, 500000)

    elif category == "Pajak & Perizinan":
        amount = np.random.lognormal(14.5, 0.8)

    elif category == "Utang & Cicilan":
        amount = np.random.lognormal(14.7, 0.9)

    elif category == "Biaya Bank":
        amount = np.random.gamma(2, 50000)

    elif category == "Lain-lain":
        amount = np.random.normal(500000, 300000)

    else:
        amount = np.random.normal(1000000, 500000)

    return int(max(5000, amount))

# =========================================================
# TYPO / NOISE GENERATOR
# =========================================================

def add_business_noise(text):

    typo_prob = 0.12

    if random.random() > typo_prob:
        return text

    typo_dict = {

        "Transfer": "Trf",
        "Pembayaran": "Pmbayaran",
        "Marketplace": "Marketplce",
        "Subscription": "Subcription",
        "Refund": "Refun",
        "Payment": "Paymnt",
        "Administrasi": "Adminstrasi"
    }

    for k, v in typo_dict.items():

        if k in text:

            return text.replace(k, v)

    return text

# =========================================================
# GENERATE DATASET
# =========================================================

def generate_business_dataset(num_rows=5000):

    data = []

    start_date = datetime.now() - timedelta(days=730)
    end_date = datetime.now()

    for i in range(num_rows):

        # =================================================
        # Generate transaction type
        # =================================================

        transaction_type = np.random.choice(
            ["Pemasukan", "Pengeluaran"],
            p=[0.55, 0.45]
        )

        # =================================================
        # Generate category
        # =================================================

        category = random.choice(
            business_transaction_types[
                transaction_type
            ]
        )

        # =================================================
        # Generate description
        # =================================================

        description = random.choice(
            business_categories[category]
        )

        # tambahkan ambiguity
        ambiguous_descriptions = [

            "Transfer",
            "Pembayaran",
            "Tagihan",
            "Marketplace",
            "Pembelian",
            "Cashback",
            "Refund",
            "QRIS",
            "Payment gateway",
            "Pembayaran digital",
            "Administrasi"
        ]

        if random.random() < 0.15:

            description = random.choice(
                ambiguous_descriptions
            )

        # typo/noise
        description = add_business_noise(
            description
        )

        # =================================================
        # Generate amount
        # =================================================

        amount = generate_business_amount(
            category
        )

        # overlap amount antar kategori
        if random.random() < 0.20:

            amount *= random.uniform(0.5, 2)

        amount = int(amount)

        # =================================================
        # Payment method
        # tidak deterministic
        # =================================================

        payment_method = np.random.choice(
            business_payment_methods,
            p=[0.08, 0.22, 0.12, 0.18, 0.28, 0.07, 0.05]
        )

        # =================================================
        # Date generation
        # =================================================

        transaction_date = fake.date_between_dates(
            date_start=start_date,
            date_end=end_date
        )

        # =================================================
        # Random category swap
        # mengurangi perfect classification
        # =================================================

        if random.random() < 0.05:

            all_categories = []

            for cats in business_transaction_types.values():

                all_categories.extend(cats)

            category = random.choice(
                all_categories
            )

        # =================================================
        # Add row
        # =================================================

        data.append({

            "transaction_id": i + 1,

            "transaction_date":
                transaction_date.strftime("%Y-%m-%d"),

            "description": description,

            "category": category,

            "transaction_type": transaction_type,

            "amount": amount,

            "payment_method": payment_method
        })

    return pd.DataFrame(data)

# =========================================================
# GENERATE DATASET
# =========================================================

business_num_rows = 5000

df_business = generate_business_dataset(
    business_num_rows
)

# =========================================================
# PREVIEW
# =========================================================

display(df_business.head())

print(df_business.info())

print("\nJumlah kategori:")
print(df_business["category"].value_counts())

,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1,2024-06-09,Pembelian,Piutang,Pemasukan,595193,Kartu Debit
1,2,2025-01-13,Pmbayaran pelanggan,Piutang,Pemasukan,2104366,Transfer Bank
2,3,2025-12-04,Pembayaran cicilan,Utang & Cicilan,Pengeluaran,1435766,Kartu Debit
3,4,2025-04-30,Pendanaan usaha,Modal & Investasi,Pemasukan,5539125,E-Wallet
4,5,2025-02-12,Refun,Penjualan,Pemasukan,2310919,Kartu Debit


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    5000 non-null   int64 
 1   transaction_date  5000 non-null   object
 2   description       5000 non-null   object
 3   category          5000 non-null   object
 4   transaction_type  5000 non-null   object
 5   amount            5000 non-null   int64 
 6   payment_method    5000 non-null   object
dtypes: int64(2), object(5)
memory usage: 273.6+ KB
None

Jumlah kategori:
category
Piutang                    936
Penjualan                  921
Modal & Investasi          882
Pembelian Stok             233
Peralatan & Aset           222
Marketing & Promosi        222
Operasional Kantor         213
Gaji & Karyawan            212
Lain-lain                  210
Transportasi & Logistik    195
Biaya Bank                 191
Pajak & Perizinan          191
Software & Langganan    

In [6]:
# --- Save Business Dataset to CSV ---
df_business.to_csv("fintrack_business_dataset.csv", index=False)
print("\nDataset saved as 'fintrack_business_dataset.csv'")


Dataset saved as 'fintrack_business_dataset.csv'


In [7]:
# =========================================================
# NOISE CONFIGURATION
# =========================================================

missing_percentage = 0.01
duplicate_percentage = 0.05
negative_amount_percentage = 0.02


# =========================================================
# ADD NOISE FUNCTION
# =========================================================

def add_noise_to_dataframe(df):

    df_noisy = df.copy()

    print("Applying noise injection...")

    # =====================================================
    # 1. MISSING VALUES
    # =====================================================

    num_rows = len(df_noisy)
    num_cols = len(df_noisy.columns)

    num_missing = int(
        num_rows * num_cols * missing_percentage
    )

    for _ in range(num_missing):

        row_idx = random.randint(0, num_rows - 1)

        col_idx = random.randint(0, num_cols - 1)

        column_name = df_noisy.columns[col_idx]

        # hindari missing pada target
        if column_name == "category":
            continue

        df_noisy.iat[row_idx, col_idx] = np.nan

    # =====================================================
    # 2. DUPLICATE ROWS
    # =====================================================

    num_duplicates = int(
        len(df_noisy) * duplicate_percentage
    )

    duplicate_rows = df_noisy.sample(
        n=num_duplicates,
        replace=True,
        random_state=42
    )

    df_noisy = pd.concat(
        [df_noisy, duplicate_rows],
        ignore_index=True
    )

    # =====================================================
    # 3. NEGATIVE AMOUNT
    # hanya pengeluaran
    # =====================================================

    positive_indices = df_noisy[
        (
            df_noisy["amount"] > 0
        ) &
        (
            df_noisy["transaction_type"]
            == "Pengeluaran"
        )
    ].index.tolist()

    num_negative = int(
        len(df_noisy) * negative_amount_percentage
    )

    selected_indices = random.sample(
        positive_indices,
        min(num_negative, len(positive_indices))
    )

    for idx in selected_indices:

        df_noisy.loc[idx, "amount"] *= -1

    # =====================================================
    # 5. TYPO NOISE
    # =====================================================

    typo_probability = 0.03

    typo_map = {

        "Transfer": "Trf",
        "Pembayaran": "Pmbayaran",
        "Marketplace": "Marketplce",
        "Refund": "Refun",
        "Administrasi": "Adminstrasi",
        "Payment": "Paymnt"
    }

    for idx in df_noisy.index:

        if random.random() < typo_probability:

            desc = str(df_noisy.loc[idx, "description"])

            for correct, typo in typo_map.items():

                if correct in desc:

                    df_noisy.loc[idx, "description"] = (
                        desc.replace(correct, typo)
                    )

    # =====================================================
    # 6. OUTLIER AMOUNT
    # =====================================================

    outlier_percentage = 0.01

    num_outliers = int(
        len(df_noisy) * outlier_percentage
    )

    outlier_indices = random.sample(
        list(df_noisy.index),
        num_outliers
    )

    for idx in outlier_indices:

        current_amount = df_noisy.loc[idx, "amount"]

        if pd.isna(current_amount):
            continue

        multiplier = random.uniform(3, 8)

        df_noisy.loc[idx, "amount"] = int(
            abs(current_amount) * multiplier
        )

    print("Noise injection complete.")

    return df_noisy

# =========================================================
# APPLY NOISE
# =========================================================

print("Applying noise to Business Dataset...")

df_business_noisy = add_noise_to_dataframe(
    df_business
)

# =========================================================
# PREVIEW
# =========================================================

print("\nDataset Preview:")
display(df_business_noisy.head())

print("\nDataset Info:")
print(df_business_noisy.info())

print("\nMissing Values:")
print(df_business_noisy.isnull().sum())

print("\nDuplicate Rows:")
print(df_business_noisy.duplicated().sum())

# =========================================================
# SAVE DATASET
# =========================================================

df_business_noisy.to_csv(
    "fintrack_business_noisy_dataset.csv",
    index=False
)

print(
    "\nDataset saved as "
    "'fintrack_business_noisy_dataset.csv'"
)

Applying noise to Business Dataset...
Applying noise injection...
Noise injection complete.

Dataset Preview:


,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1.0,2024-06-09,Pembelian,Piutang,Pemasukan,595193.0,Kartu Debit
1,2.0,2025-01-13,Pmbayaran pelanggan,Piutang,Pemasukan,2104366.0,Transfer Bank
2,3.0,2025-12-04,Pembayaran cicilan,Utang & Cicilan,Pengeluaran,1435766.0,Kartu Debit
3,4.0,2025-04-30,Pendanaan usaha,Modal & Investasi,Pemasukan,5539125.0,E-Wallet
4,5.0,2025-02-12,Refun,Penjualan,Pemasukan,2310919.0,Kartu Debit



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5250 entries, 0 to 5249
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    5198 non-null   float64
 1   transaction_date  5202 non-null   object 
 2   description       5203 non-null   object 
 3   category          5250 non-null   object 
 4   transaction_type  5197 non-null   object 
 5   amount            5190 non-null   float64
 6   payment_method    5193 non-null   object 
dtypes: float64(2), object(5)
memory usage: 287.2+ KB
None

Missing Values:
transaction_id      52
transaction_date    48
description         47
category             0
transaction_type    53
amount              60
payment_method      57
dtype: int64

Duplicate Rows:
232

Dataset saved as 'fintrack_business_noisy_dataset.csv'
